### 🔑 Views vs. copies: the rule to keep

**NumPy:** whether two arrays share memory depends on **how you indexed**, not on what the elements are (1-D numbers or 2-D rows behave the same).

| Indexing style | Examples                    | Result                         | Writing to the result…  |
| -------------- | --------------------------- | ------------------------------ | ----------------------- |
| Basic slicing  | `a[:K]`, `a[2:5]`, `a[::2]` | **View** (shares `a`'s memory) | ⚠️ changes `a` too      |
| Index array    | `a[idx]`                    | **Copy** (owns its memory)     | ✅ leaves `a` unchanged |
| Boolean mask   | `a[mask]`                   | **Copy** (owns its memory)     | ✅ leaves `a` unchanged |

**Python lists:** slicing _always_ makes a new list. Sharing happens only through **mutable elements inside it** (e.g. inner lists), because a slice is a _shallow_ copy.

**How to check:**

```python
np.shares_memory(b, a)   # True -> b and a share memory
b.base is a              # True -> b is a view borrowing a's buffer
```

**k-means takeaway:** `X` must stay fixed for the whole run, and only the centroids `mu` change. Make `mu` own its memory from the start, so a centroid update can't overwrite a data point:

```python
mu = X[rng.permutation(len(X))[:K]].copy()   # or: mu = X[:K].copy()
```


In [ ]:
import numpy as np

In [ ]:
rng = np.random.default_rng(seed=42)

In [ ]:
X = np.arange(12, dtype=float).reshape(6, 2)  # 6 points, 2 features
print(X)

In [ ]:
K = 2

In [ ]:
idx = rng.permutation(len(X))
X_shuf = X[idx]

In [ ]:
print(X_shuf)

In [ ]:
mu = X_shuf[:K]
print(mu)

In [ ]:
print(np.shares_memory(mu, X_shuf))
print(np.shares_memory(X, X_shuf))
print(np.shares_memory(mu, X))

In [ ]:
mu[0] = np.array([100.0, 100.0])
print(X_shuf[0])

In [ ]:
Y = np.arange(12, dtype=float)
idy = rng.permutation(len(Y))
print(idy)
Y_shuf = Y[idy]  # Always a copy.
mu_y = Y_shuf[:K]
print(np.shares_memory(mu_y, Y_shuf))
mu_y[0] = 65
print(Y_shuf[0])

In [ ]:
print(mu_y)
print(Y_shuf)
mu_y[:] = mu_y + 1
print(mu_y)
print(Y_shuf)
mu_y += 1
print(mu_y)
print(Y_shuf)

In [ ]:
print(mu_y)
print(Y_shuf)
print(mu_y.base is Y_shuf)  # True: mu_y borrows Y_shuf's buffer
print(Y_shuf.base)

In [ ]:
print(mu_y)
print(Y_shuf)
mu_y = np.arange(20, 32)
print(mu_y)
print(Y_shuf)
print(mu_y.base is Y_shuf)  # True: mu_y borrows Y_shuf's buffer
print(Y_shuf.base)

In [ ]:
print(mu)
print(X_shuf)

In [ ]:
mu[:] = mu + 1
print(mu)
print(X_shuf)

In [ ]:
mu += 1
print(mu)
print(X_shuf)

In [ ]:
mu[0] = np.array([-20, -40])
print(mu)
print(X_shuf)

In [ ]:
mu = np.array([-200, -40])
print(mu)
print(X_shuf)

In [ ]:
rng = np.random.default_rng(0)
X = np.arange(12, dtype=float).reshape(6, 2)
idx = rng.permutation(len(X))[:K]  # K random row indices; X itself is not shuffled
mu = X[idx]  # index-array indexing: already a copy
# or, if you shuffled and sliced:  mu = X[:K].copy()

print(np.shares_memory(mu, X))  # False